In [1]:
from string import Template
import json

import requests
import pandas as pd
import ast
import re

In [2]:
_temperature = 0
_max_tokens = -1
_stream = False

In [3]:
# Extraction of AI-specific requirements in documents for GUIDELINS
ai_specific_prompt_template = """
EXTRACT information from the following json input delimitated by ```. I want you to extract tasks on the key "AI_SPECIFIC".  
0. ALWAYS FINISH THE OUTPUT. Never send partial responses. Just send JSON format.
1. The text input is part of an international standard document, associated with the development lifecycle of AI systems. 
2. The input is in json format.
3. EXTRACT a list of guidelines based on the json input key "ai-specific". \
An text contains ai-specific text which provides guidance to adapt the process to AI systems. \
Extract guidelines based on OBJECTS and VERBS in each sentence, containing a unique pair OBJECT and VERB. \
The output have json format, adding a list of tasks based on the keys "oat". A new key should be added, named "AI_SPECIFIC". \
The output must should look like the following:
{"AI_SPECIFIC": 
 [
	{"guideline_id":int,"guideline_name":string,"guideline_description":string}]
 ]
}
The output contains a key called AI_SPECIFIC that contans a list of guidelines extracted from input json. There could be more than one guideline_id. \

Input: 
```
$ctext
```
"""

In [5]:
def send_request(message, system_msg="Always answer in rhymes."):
    # Your request payload
    url = "http://localhost:1234/v1/chat/completions"

    payload = {
        "messages": [
            { "role": "system", "content":  system_msg},
            { "role": "user", "content": message }
        ],
        "temperature": _temperature,
        "max_tokens": _max_tokens,
        "stream": _stream
    }

    headers = {
        "Content-Type": "application/json"
    }

    response = requests.post(url, headers=headers, data=json.dumps(payload))

    # Check for successful response
    if response.status_code == 200:
        return response.json()
    else:
        return {"error": "Request failed with status code " + str(response.status_code)}

In [6]:
# Function to take folder of files and a prompt template, and return a json-object of all the entities and relationships
def extract_ai_task(prompt_template, text, index):
    system_msg = "You are a helpful computer scientist and ISO standards expert who EXTRACTS information from documents."
    print(f"Extracting tasks {index}....")
    result = ""
    try:
        prompt = Template(prompt_template).substitute(ctext=text)
        result = send_request(prompt, system_msg=system_msg)
    except Exception as e:
        print(f"Error processing: {e}")
    return result

In [7]:
# Specify the Excel file path (replace 'output.xlsx' with your desired file name)
excel_file_path = '62304_5338 (activity level).xlsx'
url_excel = f'../{excel_file_path}'

# Export the DataFrame to Excel
df = pd.read_excel(url_excel, sheet_name='5338_raw_guidelines')

In [8]:
df.head(2)

,id,process_3_id,process_3_name,ai_specific,ai_stage
0,1,6.3.1,Project planning process,There are no additional activities or tasks d...,NaN
1,2,6.3.2,Project assessment and control process,There are no additional activities or tasks d...,NaN


In [9]:
df.drop(["id", "process_3_id",  "process_3_name", "ai_stage"], axis='columns', inplace=True)

In [9]:
# Remove column 'B' from DataFrame
#df = df.drop(columns=['INDEX', 'PURPOSE', 'NOTE', 'OAT'])

In [10]:
json_data = json.loads(df.to_json(orient='records'))

In [11]:
json_data[1]

{'ai_specific': ' There are no additional activities or tasks defined in the project assessment and control process. When implementing the activities and tasks in 6.3.2.2, either projects or organizations, or both should consider the following AI-specific particularities. In implementing the activity “plan for project assessment and control”, intervals can be determined (as defined in the quality management process) whether to update the AI system or the model (see ISO/IEC/IEEE 15288:2023, 6.3.2.3 and ISO/IEC/IEEE 12207:2017, 6.3.2.3). The implementation of an AI system is less predictable as a result of its iterative and experimental nature. For example, its progress cannot be reliably measured by counting the produced lines of code.'}

In [15]:
#response = extract_ai_task(ai_specific_prompt_template,json_data[1], 1)

Extracting tasks 1....


In [12]:
responses = []
print(f"Running pipeline for {len(json_data)} items in ISO International Standard")
for index, json_item in enumerate(json_data):
    responses.append(extract_ai_task(ai_specific_prompt_template,json_item, index+1))

Running pipeline for 3 items in ISO International Standard
Extracting tasks 1....
Extracting tasks 2....
Extracting tasks 3....


In [13]:
responses

[{'id': 'chatcmpl-90xx7k3ooln1tn12p7bzks',
  'object': 'chat.completion',
  'created': 1712592076,
  'model': 'C:\\Users\\cepedazk\\.cache\\lm-studio\\models\\TheBloke\\Llama-2-13B-chat-GGUF\\llama-2-13b-chat.Q5_K_M.gguf',
  'choices': [{'index': 0,
    'message': {'role': 'assistant',
     'content': '\nHere is the extracted information in JSON format:\n\n{\n"AI_SPECIFIC": [\n{\n"guideline_id": 1,\n"guideline_name": "Allow flexibility in model creation",\n"guideline_description": "Predictability of software development is already challenging and for model creation, this is even more the case. Creating a model can require AI data engineering, such as data gathering, data labelling and data pre-processing."\n},\n{\n"guideline_id": 2,\n"guideline_name": "Consider iterations of experiments",\n"guideline_description": "For a machine learning-based AI system, creating a model can require iterations of experiments and experimentation with different strategies and tactics to achieve the desir

In [55]:
# Specify the Excel file path (replace 'output.xlsx' with your desired file name)
excel_file_path = '62304_5338 (activity level).xlsx'
url_excel = f'../{excel_file_path}'

# Export the DataFrame to Excel
df_5338 = pd.read_excel(url_excel, sheet_name='5338_raw_guidelines')

In [56]:
df_5338

,id,process_3_id,process_3_name,ai_specific,ai_stage
0,1,6.3.1,Project planning process,There are no additional activities or tasks d...,NaN
1,2,6.3.2,Project assessment and control process,There are no additional activities or tasks d...,NaN
2,3,6.3.5,Configuration management process,There are no additional activities and tasks ...,NaN


In [57]:
column_names_5338 = df_5338.columns.tolist()
column_names_5338.append("guide_id")
column_names_5338.append("guide_name")
column_names_5338.append("guide_description")
print(column_names_5338)

['id', 'process_3_id', 'process_3_name', 'ai_specific', 'ai_stage', 'guide_id', 'guide_name', 'guide_description']


In [58]:
df_5338

,id,process_3_id,process_3_name,ai_specific,ai_stage
0,1,6.3.1,Project planning process,There are no additional activities or tasks d...,NaN
1,2,6.3.2,Project assessment and control process,There are no additional activities or tasks d...,NaN
2,3,6.3.5,Configuration management process,There are no additional activities and tasks ...,NaN


In [61]:
df_5338_with_guidelince = pd.DataFrame(columns=column_names_5338)

for index, response in enumerate(responses):
    # Actually row in standard
    row = df_5338.iloc[index]

    # Pattern to match inner JSON objects
    pattern = r'({\s*[^{}]+\s*})'

    # Find all matches
    
    matches = re.findall(pattern, response['choices'][0]['message']['content'], re.DOTALL)
    
    for match in matches:
        json_m = json.loads(match)
        new_row = row.copy().to_dict()
        new_row['guide_id'] = json_m['guideline_id']
        new_row['guide_name'] = json_m['guideline_name']
        new_row['guide_description'] = json_m['guideline_description']
        
        new_row = pd.DataFrame([new_row])
        df_5338_with_tasks = pd.concat([df_5338_with_tasks, new_row], )        


In [62]:
df_5338_with_tasks

,id,process_3_id,process_3_name,ai_specific,ai_stage,guide_id,guide_name,guide_description
0,1,6.3.1,Project planning process,There are no additional activities or tasks d...,NaN,1,Allow flexibility in model creation,Predictability of software development is alre...
0,1,6.3.1,Project planning process,There are no additional activities or tasks d...,NaN,2,Consider iterations of experiments,"For a machine learning-based AI system, creati..."
0,1,6.3.1,Project planning process,There are no additional activities or tasks d...,NaN,3,Acquire knowledge,"For a knowledge engineering-based AI system, c..."
0,1,6.3.1,Project planning process,There are no additional activities or tasks d...,NaN,4,Establish continuous validation,"Furthermore, project planning should take into..."
0,2,6.3.2,Project assessment and control process,There are no additional activities or tasks d...,NaN,1,Consider AI-specific particularities when impl...,The implementation of an AI system is less pre...
0,2,6.3.2,Project assessment and control process,There are no additional activities or tasks d...,NaN,2,Determine intervals for updating AI systems or...,Intervals can be determined (as defined in the...
0,3,6.3.5,Configuration management process,There are no additional activities and tasks ...,NaN,1,Consider AI-specific particularities in config...,When implementing the activities and tasks in ...
0,3,6.3.5,Configuration management process,There are no additional activities and tasks ...,NaN,2,Manage AI-specific artefacts,"The artefacts of AI systems, especially datase..."
0,3,6.3.5,Configuration management process,There are no additional activities and tasks ...,NaN,3,Rollback to a previous version of a runtime model,A typical application of configuration managem...
0,3,6.3.5,Configuration management process,There are no additional activities and tasks ...,NaN,4,Manage data used to build and test the AI model,The data that are used to build and test the A...


In [64]:
# Specify the Excel file path (replace 'output.xlsx' with your desired file name)
excel_file_path = 'output-5338_guidelines.xlsx'

# Export the DataFrame to Excel
df_5338_with_tasks.to_excel(excel_file_path, index=False)

In [29]:
with open("responses_llm_05-03-2024.txt", 'w') as f:
    f.write(json.dumps(responses))